# Фракталы как вычислительный эксперимент

В зелёном уровне построили множество Мандельброта и заполненные множества Джулиа для отображения
$$z_{n+1}=z_n^2+c.$$
Теперь увеличим участок границы, отдельно проверим влияние сетки и числа итераций, сравним две окраски и построим анимацию при изменении $c$.

Для Мандельброта меняется параметр $c$, начальная точка — $z_0=0$. Для заполненного множества Джулиа $K_c$ меняется $z_0$, а $c$ фиксирован. Собственно множество Джулиа — граница $J_c=\partial K_c$.

Все изображения ниже получаются новым расчётом на комплексной сетке. Чёрный цвет означает только отсутствие выхода за проверенное число шагов.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm, Normalize
from matplotlib.patches import Rectangle, Patch
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display
from pathlib import Path
from time import perf_counter
import csv

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10
plt.rcParams["animation.embed_limit"] = 80

output_dir = Path("3.3_yellow_melnikov_artifacts")
output_dir.mkdir(exist_ok=True)

cmap = plt.get_cmap("turbo").copy()
cmap.set_bad("black")
no_escape = Patch(facecolor="black", label="Нет выхода за Nmax шагов")
print("Все результаты отображаются непосредственно в ноутбуке.")


## 1. Сетка и первая итерация выхода

Используем булеву маску активных точек из зелёного уровня. После строгого выхода $|z_n|>R$ точка больше не обновляется. Вместе с номером $n$ запоминаем модуль $|z_n|$ — он понадобится для плавной окраски.

Для Мандельброта берём $R=2$, для Джулиа — $R=\max(2,|c|)$. Если начальная точка уже находится за радиусом, номер выхода равен нулю. Значение $N_{\max}+1$ — служебный признак отсутствия выхода.

Точки сетки лежат в центрах пикселей. При ширине области $L$ шаг равен $L/m$ для сетки $m\times m$. Границы области точно совпадают с границами изображения.


In [ ]:
def complex_grid(extent, size):
    xmin, xmax, ymin, ymax = extent
    x = xmin + (np.arange(size) + 0.5) * (xmax - xmin) / size
    y = ymin + (np.arange(size) + 0.5) * (ymax - ymin) / size
    return x[np.newaxis, :] + 1j * y[:, np.newaxis]


def escape_iterations(z0, c, Nmax, R=2):
    initial, parameters = np.broadcast_arrays(
        np.asarray(z0, dtype=complex), np.asarray(c, dtype=complex)
    )
    shape = initial.shape
    z = initial.reshape(-1).copy()
    c_values = parameters.reshape(-1)
    active = np.ones(z.size, dtype=bool)
    result = np.full(z.size, Nmax + 1, dtype=np.int32)
    exit_modulus = np.full(z.size, np.nan)

    initial_modulus = np.abs(z)
    escaped = initial_modulus > R
    result[escaped] = 0
    exit_modulus[escaped] = initial_modulus[escaped]
    active[escaped] = False

    for n in range(1, Nmax + 1):
        indices = np.flatnonzero(active)
        if indices.size == 0:
            break
        z[indices] = z[indices]**2 + c_values[indices]
        modulus = np.abs(z[indices])
        escaped_here = modulus > R
        exited = indices[escaped_here]
        result[exited] = n
        exit_modulus[exited] = modulus[escaped_here]
        active[exited] = False

    return result.reshape(shape), exit_modulus.reshape(shape)


def mandelbrot(extent, size, Nmax):
    return escape_iterations(0j, complex_grid(extent, size), Nmax, R=2)


def integer_colors(result, Nmax):
    return np.ma.masked_where(result > Nmax, result)


def draw_fractal(ax, values, extent, title, norm, plane="c"):
    image = ax.imshow(
        values, origin="lower", extent=extent, aspect="equal",
        cmap=cmap, norm=norm, interpolation="nearest"
    )
    ax.set_title(title)
    ax.set_xlabel(f"Re {plane}")
    ax.set_ylabel(f"Im {plane}")
    return image


## 2. Увеличение выбранного участка границы

Выбираем квадрат
$$-0.80\leq\operatorname{Re}c\leq-0.70,\qquad
0.05\leq\operatorname{Im}c\leq0.15.$$
Он находится у границы основной кардиоиды и периода-2 компоненты. На общей карте отмечаем его белой рамкой. Ширина окна уменьшается с $3$ до $0.1$, то есть увеличение по каждой оси составляет $30$ раз.

Общую карту используем для выбора места; увеличенный фрагмент пересчитываем заново, а не растягиваем её пиксели.


In [ ]:
full_extent = [-2, 1, -1.5, 1.5]
zoom_extent = [-0.80, -0.70, 0.05, 0.15]
overview, overview_modulus = mandelbrot(full_extent, 600, 200)
zoom_preview, zoom_preview_modulus = mandelbrot(zoom_extent, 600, 200)

preview_norm = PowerNorm(gamma=0.4, vmin=0, vmax=200)
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
draw_fractal(ax[0], integer_colors(overview, 200), full_extent,
             "Общая карта: 600×600, Nmax=200", preview_norm)
image = draw_fractal(ax[1], integer_colors(zoom_preview, 200), zoom_extent,
                     "Новый расчёт выбранного окна: ×30", preview_norm)
ax[0].add_patch(Rectangle((zoom_extent[0], zoom_extent[2]), 0.1, 0.1,
                          fill=False, edgecolor="white", linewidth=1.5))
ax[0].annotate("Выбранное окно", (-0.75, 0.15), xytext=(-1.8, 0.8),
               color="white", arrowprops=dict(arrowstyle="->", color="white"))
bar = fig.colorbar(image, ax=ax, shrink=0.78, ticks=[0, 10, 25, 50, 100, 200])
bar.set_label("Первая итерация выхода n")
fig.legend(handles=[no_escape], loc="outside lower center")
fig.savefig(output_dir / "01_boundary_zoom.png", dpi=180)
plt.show()


## 3. Разрешение и число итераций

Строим один фрагмент для четырёх сочетаний: $600\times600$ и $1200\times1200$, $N_{\max}=200$ и $800$. Строки отличаются разрешением, столбцы — числом итераций.

Во всех четырёх панелях **одна шкала от 0 до 800** и одна палитра. Чтобы были видны и ранние, и поздние выходы, используем одинаковое преобразование цвета $q=(n/800)^{0.4}$. Деления цветовой шкалы подписаны самими номерами итераций. Масштаб осей равный, интерполяция пикселей отключена.


In [ ]:
sizes = [600, 1200]
limits = [200, 800]
results = {}
timings = {}

for size in sizes:
    for Nmax in limits:
        start = perf_counter()
        results[size, Nmax] = mandelbrot(zoom_extent, size, Nmax)
        timings[size, Nmax] = perf_counter() - start
        print(f"{size}×{size}, Nmax={Nmax}: {timings[size, Nmax]:.2f} с")

common_norm = PowerNorm(gamma=0.4, vmin=0, vmax=800)
fig, ax = plt.subplots(2, 2, figsize=(11, 10), layout="constrained")
fig.suptitle("Один участок границы: отдельное влияние сетки и Nmax")
for row, size in enumerate(sizes):
    for col, Nmax in enumerate(limits):
        result, _ = results[size, Nmax]
        image = draw_fractal(ax[row, col], integer_colors(result, Nmax), zoom_extent,
                             f"{size}×{size}, Nmax={Nmax}", common_norm)
bar = fig.colorbar(image, ax=ax, shrink=0.8,
                   ticks=[0, 10, 25, 50, 100, 200, 400, 800])
bar.set_label("Первая итерация выхода n; общая шкала")
fig.legend(handles=[no_escape], loc="outside lower center")
fig.savefig(output_dir / "02_resolution_iterations.png", dpi=220)
plt.show()


In [ ]:
statistics = []
print(" Сетка    Nmax    Шаг по осям     Без выхода, %    Выход на шагах 201–800, %")
for size in sizes:
    long_result = results[size, 800][0]
    delayed = np.count_nonzero((long_result > 200) & (long_result <= 800))
    for Nmax in limits:
        result = results[size, Nmax][0]
        pending = np.count_nonzero(result > Nmax)
        row = dict(size=size, Nmax=Nmax, step=0.1 / size,
                   unresolved=pending, unresolved_percent=100 * pending / result.size,
                   delayed_percent=100 * delayed / result.size)
        statistics.append(row)
        print(f"{size:5d}    {Nmax:4d}    {row['step']:.7f}       "
              f"{row['unresolved_percent']:8.4f}            {row['delayed_percent']:8.4f}")

    # На одной и той же сетке ранние выходы должны полностью совпасть.
    short_result = results[size, 200][0]
    early = short_result <= 200
    assert np.array_equal(short_result[early], long_result[early])
    assert np.count_nonzero(short_result > 200) - np.count_nonzero(long_result > 800) == delayed

with (output_dir / "resolution_statistics.csv").open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(statistics[0]))
    writer.writeheader()
    writer.writerows(statistics)

for size in sizes:
    result = results[size, 800][0]
    delayed = np.count_nonzero((result > 200) & (result <= 800))
    print(f"На сетке {size}×{size} из чёрной области при Nmax=200 "
          f"ушли {delayed} точек ({100 * delayed / result.size:.4f}% всей сетки).")


При фиксированном $N_{\max}$ переход от $600$ к $1200$ уменьшает пространственный шаг вдвое и увеличивает число точек вчетверо. Лучше различаются тонкие полосы и мелкие детали, но время наблюдения каждой орбиты остаётся прежним.

При фиксированной сетке переход от $200$ к $800$ не создаёт новых пространственных точек. Он обнаруживает поздние выходы: часть ранее чёрных точек получает цвет. Число таких точек указано в таблице и выводе расчёта. Доля чёрных пикселей — характеристика этой сетки и этого ограничения, а не точное измерение площади множества.

Сетки из центров пикселей при удвоении разрешения не вложены. Поэтому поэлементно сравниваем только расчёты с одинаковым разрешением; разные сетки сравниваем по изображению, шагу и долям точек.

В этом окне доля точек без выхода при увеличении $N_{\max}$ уменьшилась с **74.2183% до 71.5200%** на сетке $600\times600$ и с **74.2171% до 71.5090%** на сетке $1200\times1200$. Поздние выходы составили соответственно **9714** и **38997** точек — около **2.7%** всей сетки.


## 4. Обычная и плавная окраска

Обычная окраска использует целый номер первой итерации выхода $n$. Все точки с одинаковым $n$ имеют один цвет, поэтому снаружи появляются полосы.

Для плавной окраски используем модуль орбиты **именно в момент первого выхода**:
$$\nu=n-\frac{\ln\!\left(\ln|z_n|/\ln R\right)}{\ln2},\qquad |z_n|>R.$$
Далеко от множества $|z_{n+1}|\approx|z_n|^2$, следовательно, $\ln|z_{n+k}|\approx2^k\ln|z_n|$. Если продолжить эту зависимость на дробное число шагов до уровня $\ln R$, получаем указанную формулу. При $|z_n|=R$ она даёт $\nu=n$. Чем дальше точка перелетела порог, тем меньше дробная оценка времени выхода.

Часто используют $n+1-\ln(\ln|z_n|)/\ln2$. Это та же логарифмическая поправка с другим постоянным сдвигом шкалы: здесь отсчёт привязан к радиусу $R$. Формула приближённая, поскольку слагаемое $c$ при выходе ещё может быть заметно. Она смягчает цветовые полосы, но не повышает разрешение и не уточняет принадлежность множеству.

Чтобы различие полос было заметнее, только в следующем сравнении сужаем общую для двух панелей шкалу до $10\ldots80$. Значения ниже 10 и выше 80 получают крайние цвета (треугольники на шкале); чёрная маска не меняется. Для исследования четырёх сочетаний выше остаётся полный диапазон $0\ldots800$.


In [ ]:
def smooth_colors(result, exit_modulus, Nmax, R=2):
    escaped = result <= Nmax
    values = np.zeros(result.shape, dtype=float)
    values[escaped] = result[escaped] - np.log2(
        np.log(exit_modulus[escaped]) / np.log(R)
    )
    return np.ma.array(values, mask=~escaped)


result, modulus = results[1200, 800]
smooth_result = smooth_colors(result, modulus, 800)
# Сужаем только цветовую шкалу, чтобы отчётливее различались полосы.
coloring_norm = Normalize(vmin=10, vmax=80, clip=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 5.2), layout="constrained")
fig.suptitle("Одна сетка 1200×1200, Nmax=800; одинаковая палитра и шкала")
draw_fractal(ax[0], integer_colors(result, 800), zoom_extent,
             "Обычная окраска: целое n", coloring_norm)
image = draw_fractal(ax[1], smooth_result, zoom_extent,
                     "Плавная окраска: дробное ν", coloring_norm)
bar = fig.colorbar(image, ax=ax, shrink=0.8,
                   ticks=[10, 20, 30, 40, 60, 80], extend="both")
bar.set_label("Число итераций: n или ν")
fig.legend(handles=[no_escape], loc="outside lower center")
fig.savefig(output_dir / "03_integer_vs_smooth.png", dpi=220)
plt.show()

assert np.array_equal(np.ma.getmaskarray(smooth_result), result > 800)
assert np.isfinite(smooth_result.compressed()).all()


На левом изображении соседние полосы соответствуют разным целым $n$. Справа внутри каждой полосы учитывается величина перелёта за $R$, и переходы становятся мягче. Чёрные области полностью совпадают: меняется только окраска уже вышедших точек. Интерполяция изображения по-прежнему отключена.


## 5. Движение параметра через границу Мандельброта

Выбираем прямой отрезок вещественной оси:
$$c(t)=0.05+0.40t,\qquad 0\leq t\leq1.$$
Он пересекает границу при $c=1/4$ ($t=1/2$). До этой точки путь лежит в основной кардиоиде; после неё вещественная критическая орбита уходит на бесконечность.

Это пересечение можно обосновать без картинки. При $0\leq c\leq1/4$ существует неподвижная точка
$$\alpha=\frac{1-\sqrt{1-4c}}2.$$
Итерации из нуля монотонно растут и остаются в $[0,\alpha]$. При $c>1/4$ имеем
$$x_{n+1}-x_n=(x_n-1/2)^2+c-1/4>0.$$
Если бы возрастающая орбита была ограничена, её предел был бы вещественной неподвижной точкой, но таких точек при $c>1/4$ нет.

Берём 81 кадр, включая точное $c=1/4$ в кадре с индексом 40. На нескольких кадрах отдельно считаем критическую орбиту до 5000 шагов и останавливаемся сразу после выхода.


In [ ]:
frame_t = np.linspace(0, 1, 81)
c_path = (0.05 + 0.40 * frame_t).astype(complex)
c_path[40] = 0.25 + 0j
critical_Nmax = 5000


def critical_orbit(c, Nmax=5000):
    R = max(2, abs(c))
    orbit = [0j]
    z = 0j
    for n in range(1, Nmax + 1):
        z = z**2 + c
        orbit.append(z)
        if abs(z) > R:
            return np.asarray(orbit), n
    return np.asarray(orbit), None


critical_results = [critical_orbit(c, critical_Nmax) for c in c_path]
selected_frames = [0, 20, 40, 41, 50, 80]
critical_rows = []
print("Кадр     c       Первый выход     max |z_n|       Последнее Re z_n")
for frame in selected_frames:
    c = c_path[frame]
    orbit, first_exit = critical_results[frame]
    status = str(first_exit) if first_exit is not None else f"нет за {critical_Nmax}"
    print(f"{frame:3d}    {c.real:.3f}    {status:>12s}       "
          f"{np.max(np.abs(orbit)):.6f}          {orbit[-1].real:.6f}")
    critical_rows.append(dict(frame=frame, c_real=c.real, c_imag=c.imag,
                              first_exit=first_exit, max_modulus=np.max(np.abs(orbit)),
                              last_real=orbit[-1].real))

with (output_dir / "critical_orbits.csv").open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(critical_rows[0]))
    writer.writeheader()
    writer.writerows(critical_rows)

fig, ax = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for frame in selected_frames:
    orbit, first_exit = critical_results[frame]
    label = f"кадр {frame}, c={c_path[frame].real:.3f}"
    indices = np.arange(min(len(orbit), 121))
    target = ax[0] if first_exit is None else ax[1]
    target.plot(indices, np.abs(orbit[:len(indices)]), label=label)
ax[0].set_title("Орбиты без выхода: первые 120 шагов")
ax[1].set_title("Орбиты с выходом: до первого |z| > 2")
ax[0].axhline(0.5, linestyle=":", color="gray", label="z = 1/2 при c = 1/4")
ax[1].axhline(2, linestyle="--", color="black", label="R = 2")
for a in ax:
    a.set_xlabel("Номер итерации n")
    a.set_ylabel("|z_n|, z₀ = 0")
    a.grid(True, alpha=0.3)
    a.legend(fontsize=8)
fig.savefig(output_dir / "04_critical_orbits.png", dpi=180)
plt.show()


При $c=0.05$ и $0.15$ критическая орбита приближается к притягивающей неподвижной точке. При $c=0.25$ она тоже ограничена и стремится к $1/2$, но сходимость медленная: это параболическая граница. Для $c=0.255$, $0.30$ и $0.45$ расчёт обнаруживает выход. Чем ближе внешний параметр к $1/4$, тем дольше может длиться задержка перед выходом.

Само отсутствие выхода за 5000 шагов не доказывает ограниченность. Для внутренних параметров и граничного $c=1/4$ здесь есть дополнительное аналитическое обоснование из предыдущего раздела.


## 6. Анимация: параметр и соответствующее множество Джулиа

Слева показываем карту параметров и движущуюся точку $c$, справа — заполненное множество Джулиа для **того же кадра**. Стрелка показывает направление пути. В обоих изображениях используем плавную окраску, общий диапазон $0\ldots400$ и неизменный масштаб осей. Чёрный цвет — отсутствие выхода за 400 шагов.

Карта параметров вычисляется один раз на сетке $700\times700$. Каждый кадр Джулиа пересчитывается на сетке $480\times480$; эти настройки отдельно от исследования сеток выше. Для каждого кадра также выводим результат проверки критической орбиты до 5000 шагов.

Анимация с ползунком и кнопками воспроизведения встроена в вывод следующей ячейки через `HTML(animation.to_jshtml())`, как в работе про маятник. Все 81 кадр находятся внутри `.ipynb`; для передачи работы достаточно одного этого файла. Виджеты `ipywidgets`, отдельный HTML и файлы на компьютере автора не используются.

Если среда просмотра блокирует JavaScript в полученном ноутбуке, нужно разрешить доверенный вывод (Trust Notebook / доверие рабочей области) либо выполнить ячейки сверху вниз. Для повторного расчёта нужны NumPy, Matplotlib, Pillow и Jupyter/IPython. При запуске GIF и изображения сохраняются в новую папку `3.3_yellow_melnikov_artifacts` в текущем рабочем каталоге; для просмотра уже сохранённой анимации эта папка не нужна.


In [ ]:
parameter_extent = [-0.6, 0.6, -0.6, 0.6]
julia_extent = [-1.5, 1.5, -1.5, 1.5]
animation_size = 480
animation_Nmax = 400
fps = 12
animation_norm = PowerNorm(gamma=0.4, vmin=0, vmax=animation_Nmax)

parameter_result, parameter_modulus = mandelbrot(parameter_extent, 700, animation_Nmax)
parameter_colors = smooth_colors(parameter_result, parameter_modulus, animation_Nmax)
z_grid = complex_grid(julia_extent, animation_size)
julia_frames = []
for frame, c in enumerate(c_path):
    R = max(2, abs(c))
    result, modulus = escape_iterations(z_grid, c, animation_Nmax, R)
    # float32 достаточно только для цветов; сами орбиты считаем в complex128.
    julia_frames.append(smooth_colors(result, modulus, animation_Nmax, R).astype(np.float32))
    if frame % 20 == 0:
        print(f"Вычислен кадр {frame + 1}/{len(c_path)}, c={c.real:.3f}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.4, 5.2))
fig.subplots_adjust(left=0.07, right=0.89, bottom=0.19, top=0.86, wspace=0.30)
fig.suptitle("Изменение множества Джулиа при пересечении границы Мандельброта", fontsize=12)
draw_fractal(ax[0], parameter_colors, parameter_extent,
             "Карта параметров: 700×700", animation_norm)
ax[0].plot(c_path.real, c_path.imag, color="white", linewidth=1.2)
ax[0].annotate("", xy=(0.45, 0.0), xytext=(0.34, 0.0),
               arrowprops=dict(arrowstyle="->", color="white", lw=1.5))
ax[0].plot([0.25], [0], "+", color="cyan", markersize=11, markeredgewidth=1.5)
ax[0].annotate("c = 1/4", (0.25, 0), xytext=(0.07, 0.14), color="white",
               arrowprops=dict(arrowstyle="-", color="white"), fontsize=9)
marker, = ax[0].plot([], [], "o", color="gold", markeredgecolor="black", markersize=7)
julia_image = draw_fractal(ax[1], julia_frames[0], julia_extent,
                           "Заполненное множество Джулиа: 480×480", animation_norm, plane="z₀")
cax = fig.add_axes([0.92, 0.24, 0.018, 0.53])
bar = fig.colorbar(julia_image, cax=cax, ticks=[0, 10, 25, 50, 100, 200, 400])
bar.set_label("Плавное число итераций ν", fontsize=9)
frame_label = fig.text(0.07, 0.10, "", fontsize=10)
critical_label = fig.text(0.07, 0.055, "", fontsize=9)
fig.text(0.07, 0.015, "Обе панели: Nmax=400, R=2. Чёрный: нет выхода за 400 шагов.", fontsize=9)


def update(frame):
    c = c_path[frame]
    marker.set_data([c.real], [c.imag])
    julia_image.set_data(julia_frames[frame])
    frame_label.set_text(f"Кадр {frame:02d}/80; t={frame_t[frame]:.4f}; c={c.real:.3f} + {c.imag:.3f}i")
    orbit, first_exit = critical_results[frame]
    if first_exit is None:
        status = f"нет выхода за {critical_Nmax} шагов; |z_last|={abs(orbit[-1]):.6f}"
    else:
        status = f"выход на шаге {first_exit}; |z_exit|={abs(orbit[-1]):.6f}"
    critical_label.set_text("Критическая орбита z₀=0: " + status)
    return marker, julia_image, frame_label, critical_label


update(40)
fig.savefig(output_dir / "05_animation_boundary_frame.png", dpi=160)
animation = FuncAnimation(fig, update, frames=len(c_path), interval=1000 / fps, blit=False)
gif_path = output_dir / "julia_parameter.gif"
animation.save(gif_path, writer=PillowWriter(fps=fps), dpi=90)


plt.close(fig)

# HTML содержит сами PNG-кадры в base64, а не ссылки на файлы.
animation_html = animation.to_jshtml(fps=fps, default_mode="once")

# Заменяем внешнюю библиотеку значков обычными символами:
# управление не требует загрузки CSS или шрифтов из интернета.
import re
animation_html = re.sub(r'<link\b[^>]*font-awesome[^>]*>', '', animation_html,
                        flags=re.DOTALL)
button_icons = {
    "fa fa-minus": "−", "fa fa-fast-backward": "⏮",
    "fa fa-step-backward": "⏪", "fa fa-play fa-flip-horizontal": "◀",
    "fa fa-pause": "⏸", "fa fa-play": "▶",
    "fa fa-step-forward": "⏩", "fa fa-fast-forward": "⏭",
    "fa fa-plus": "+",
}
for css_class, symbol in button_icons.items():
    animation_html = animation_html.replace(
        f'<i class="{css_class}"></i>', f'<span>{symbol}</span>'
    )
display(HTML(animation_html))
print("81 кадр встроен в ноутбук. Ползунок выбирает кадр; ▶ запускает анимацию.")


## 7. Ограниченность критической орбиты и связность

Для квадратичного отображения $z\mapsto z^2+c$ справедлив критерий:
$$c\in M
\quad\Longleftrightarrow\quad
\{f_c^{\,n}(0)\}_{n\geq0}\ \text{ограничена}
\quad\Longleftrightarrow\quad
K_c\ \text{связно}
\quad\Longleftrightarrow\quad
J_c\ \text{связно}.$$
Точка $0$ критическая, поскольку $f'_c(z)=2z$. Если её орбита уходит на бесконечность, то для этой квадратичной семьи $K_c=J_c$ — вполне несвязное канторово множество. Пустой чёрный участок растровой картинки не означает, что само множество пусто.

На выбранном пути до $c=1/4$ и в самой граничной точке множества связны. После пересечения границы они вполне несвязны. Вблизи границы изменение связности может быть трудно заметить на кадрах.

**Какие ограничения накладывает конечная картинка?**

- Конечный $N_{\max}$ оставляет среди чёрных точек те, которые выйдут позже. Это может создавать ложные перемычки и завышать видимую область.
- Конечный шаг сетки может пропустить тонкую перемычку связного множества или малый промежуток между компонентами несвязного. По соседству окрашенных пикселей нельзя доказывать топологическую связность.
- Увеличение разрешения и увеличение $N_{\max}$ устраняют разные источники ошибки; одно не заменяет другое. Здесь эти эффекты разделены четырьмя расчётами.
- Влияют и ошибки арифметики с плавающей точкой, особенно у неустойчивых орбит. Больше итераций само по себе не даёт строгого доказательства на границе.

Обнаруженный выход критической орбиты — надёжный численный признак внешнего параметра; строгое машинное доказательство в пограничных случаях потребовало бы контроля округления. Отсутствие выхода за конечное время остаётся только результатом наблюдения. Для нашего вещественного пути связность дополнительно установлена аналитически, а не по виду растра.
